In [2]:
import math
from sec_lib import (
    InputModule, OutputModule, ProcessingModule,
    Architecture, build_architecture,
    SimpleTask, SplittingTask, MergingTask,
    Algorithm, run_algorithm,
    mapping_cost, routing_cost, total_cost,
    processing_time_cost, processing_energy_cost,
)


## 1. Scenario: On-Board vs ISL Downlink

Two architectures are compared for a single-task observation pipeline:

| Architecture | Processing | Output link | Link length |
|---|---|---|---|
| **Arch A** | OBDH | Feeder Link (ground station) | 800 km |
| **Arch B** | OBDH | ISL (nearby satellite) | 200 km |

Both architectures run the same single-task algorithm that compresses the raw
image data before downlinking. The only difference is where the data ends up
and the length of the outgoing link.

In [3]:
# ── Architectures ─────────────────────────────────────────────────────────────

base_config = {
    "OBDH":       {
        "ew": 1e-6,  "eup": 30,  "speed": 3.6e12,
        "connections" : [{"to": "FeederLink"}]
    },
    "FeederLink": {"energy": 15, "speed": 10e6, "link_length": 800},
}

config_A = {
    "OBDH":       {
        "ew": 1e-6,  "eup": 30,  "speed": 3.6e12,
        "connections" : [{"to": "ISL1_out"}]
    },
    "ISL1_out":  {
        "energy": 15, "speed": 500e6, "link_length": 100,
        "connections" : [{"to": "ISL1_in"}]
    },
    "ISL1_in": {
        "energy": 1e-6,
        "connections" : [{"to": "EC"}]
    },
    "EC":       {
        "ew": 1e-6,  "eup": 30,  "speed": 3.6e12,
        "connections" : [{"to": "FeederLink"}]
    },
    "FeederLink":  {"energy": 15, "speed": 500e6, "link_length": 100},
}

config_B = {
    "OBDH":       {
        "ew": 1e-6,  "eup": 30,  "speed": 3.6e12,
        "connections" : [{"to": "ISL1_out"}]
    },
    "ISL1_out":  {
        "energy": 15, "speed": 500e6, "link_length": 100,
        "connections" : [{"to": "ISL1_in"}]
    },
    "ISL1_in" : {
        "energy": 1e-6,
        "connections" : [{"to": "EC"}]
    },
    "EC":       {
        "ew": 1e-6,  "eup": 30,  "speed": 3.6e12,
        "connections" : [{"to": "ISL2_out"}]
    },
    "ISL2_out":  {
        "energy": 15, "speed": 500e6, "link_length": 100,
        "connections" : [{"to": "ISL2_in"}]
    },
    "ISL2_in" : {"energy": 1e-6},
}

arch_base = build_architecture(base_config)
arch_A = build_architecture(config_A)
arch_B = build_architecture(config_B)

print("Architecture Base:", {n: type(m).__name__ for n, m in arch_base.modules.items()})
print("Architecture A:", {n: type(m).__name__ for n, m in arch_A.modules.items()})
print("Architecture B:", {n: type(m).__name__ for n, m in arch_B.modules.items()})

Architecture Base: {'OBDH': 'ProcessingModule', 'FeederLink': 'OutputModule'}
Architecture A: {'OBDH': 'ProcessingModule', 'ISL1_out': 'OutputModule', 'ISL1_in': 'InputModule', 'EC': 'ProcessingModule', 'FeederLink': 'OutputModule'}
Architecture B: {'OBDH': 'ProcessingModule', 'ISL1_out': 'OutputModule', 'ISL1_in': 'InputModule', 'EC': 'ProcessingModule', 'ISL2_out': 'OutputModule', 'ISL2_in': 'InputModule'}


In [ ]:
# ── Algorithm ─────────────────────────────────────────────────────────────────

process1_task = SimpleTask(name="process1", phi=0.9,  n_op=lambda d: d * 600)
process2_task = SimpleTask(name="process2", phi=0.75, n_op=lambda d: d * 100)

algo_scenario = Algorithm()
algo_scenario.add(process1_task)
algo_scenario.add(process2_task)
algo_scenario.connect("process1", "process2")

d_start_scenario = 100_000_000  # 100 MB raw image in bits

algo_scenario = run_algorithm(algo_scenario, d_start_scenario)

print(f"d_in  process1 : {algo_scenario.d_in('process1'):.3e} bits")
print(f"d_in  process2 : {algo_scenario.d_in('process2'):.3e} bits")
print(f"d_out process2 : {process2_task.phi * algo_scenario.d_in('process2'):.3e} bits")
print(f"workload process1 : {algo_scenario.workloads['process1']:.3e} ops")
print(f"workload process2 : {algo_scenario.workloads['process2']:.3e} ops")

In [ ]:
# ── Mapping ────────────────────────────────────────────────────────────────────

obdh_base = arch_base["OBDH"]
obdh_A    = arch_A["OBDH"]
obdh_B    = arch_B["OBDH"]
ec_A      = arch_A["EC"]
ec_B      = arch_B["EC"]

# Base: process1 + process2 on OBDH, downlink via FeederLink
mapping_base = {
    "process1": (obdh_base, algo_scenario.d_in("process1"), algo_scenario.workloads["process1"]),
    "process2": (obdh_base, algo_scenario.d_in("process2"), algo_scenario.workloads["process2"]),
}

# Arch A: process1 on OBDH, process2 on EC (after ISL hop), downlink via FeederLink
mapping_A = {
    "process1": (obdh_A, algo_scenario.d_in("process1"), algo_scenario.workloads["process1"]),
    "process2": (ec_A,   algo_scenario.d_in("process2"), algo_scenario.workloads["process2"]),
}

# Arch B: process1 on OBDH, process2 on EC (after ISL hop), downlink via second ISL
mapping_B = {
    "process1": (obdh_B, algo_scenario.d_in("process1"), algo_scenario.workloads["process1"]),
    "process2": (ec_B,   algo_scenario.d_in("process2"), algo_scenario.workloads["process2"]),
}

# ── Routing ────────────────────────────────────────────────────────────────────
# data_sizes[("process1","process2")] is d_out of process1 = d_in of process2

d_after_p1 = algo_scenario.data_sizes[("process1", "process2")]
d_after_p2 = process2_task.phi * algo_scenario.d_in("process2")   # terminal task

feeder_base = arch_base["FeederLink"]
isl1_out_A  = arch_A["ISL1_out"]
isl1_in_A   = arch_A["ISL1_in"]
feeder_A    = arch_A["FeederLink"]
isl1_out_B  = arch_B["ISL1_out"]
isl1_in_B   = arch_B["ISL1_in"]
isl2_out_B  = arch_B["ISL2_out"]
isl2_in_B   = arch_B["ISL2_in"]

routing_base = {
    ("process2", "sink"): (feeder_base, d_after_p2),
}

routing_A = {
    ("process1", "EC"):   (isl1_out_A, d_after_p1),   # OBDH → ISL → EC
    ("ISL1_in",  "EC"):   (isl1_in_A,  d_after_p1),
    ("process2", "sink"): (feeder_A,   d_after_p2),
}

routing_B = {
    ("process1", "EC"):   (isl1_out_B, d_after_p1),   # OBDH → ISL1 → EC
    ("ISL1_in",  "EC"):   (isl1_in_B,  d_after_p1),
    ("process2", "sink"): (isl2_out_B, d_after_p2),   # EC → ISL2 → sink
    ("ISL2_out", "sink"): (isl2_in_B,  d_after_p2),
}

# ── Costs ──────────────────────────────────────────────────────────────────────
time_base,  energy_base  = total_cost(mapping_base, routing_base)
time_A,     energy_A     = total_cost(mapping_A,    routing_A)
time_B,     energy_B     = total_cost(mapping_B,    routing_B)

map_t_base, map_e_base   = mapping_cost(mapping_base)
rout_t_base, rout_e_base = routing_cost(routing_base)
map_t_A,    map_e_A      = mapping_cost(mapping_A)
rout_t_A,   rout_e_A     = routing_cost(routing_A)
map_t_B,    map_e_B      = mapping_cost(mapping_B)
rout_t_B,   rout_e_B     = routing_cost(routing_B)

print(f"{'':30s}  {'Base (FL)':>14s}  {'Arch A (ISL+FL)':>15s}  {'Arch B (ISL+ISL)':>16s}")
print(f"{'─'*80}")
print(f"{'Mapping   time  (s)':30s}  {map_t_base:>14.4f}  {map_t_A:>15.4f}  {map_t_B:>16.4f}")
print(f"{'Mapping   energy (J)':30s}  {map_e_base:>14.4f}  {map_e_A:>15.4f}  {map_e_B:>16.4f}")
print(f"{'Routing   time  (s)':30s}  {rout_t_base:>14.4f}  {rout_t_A:>15.4f}  {rout_t_B:>16.4f}")
print(f"{'Routing   energy (J)':30s}  {rout_e_base:>14.6f}  {rout_e_A:>15.6f}  {rout_e_B:>16.6f}")
print(f"{'─'*80}")
print(f"{'TOTAL     time  (s)':30s}  {time_base:>14.4f}  {time_A:>15.4f}  {time_B:>16.4f}")
print(f"{'TOTAL     energy (J)':30s}  {energy_base:>14.4f}  {energy_A:>15.4f}  {energy_B:>16.4f}")

In [ ]:
# ── Bar chart comparison ───────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

labels   = ["Arch A\n(Feeder Link 800 km)", "Arch B\n(ISL 200 km)"]
times    = [time_A,   time_B]
energies = [energy_A, energy_B]

x   = np.arange(len(labels))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# — time —
bars1 = ax1.bar(x, times, color=["steelblue", "darkorange"], width=0.5)
ax1.set_xticks(x); ax1.set_xticklabels(labels)
ax1.set_ylabel("Total time (s)")
ax1.set_title("Latency comparison")
ax1.bar_label(bars1, fmt="%.4f", padding=3)
ax1.grid(axis="y", alpha=0.3)

# stacked breakdown: mapping vs routing
map_e  = [map_e_A,  map_e_B]
rout_e = [rout_e_A, rout_e_B]
bars_m = ax2.bar(x, map_e,  label="Mapping",  color=["steelblue",  "darkorange"],  width=0.5)
bars_r = ax2.bar(x, rout_e, label="Routing",  color=["lightsteelblue", "moccasin"], width=0.5,
                  bottom=map_e)
ax2.set_xticks(x); ax2.set_xticklabels(labels)
ax2.set_ylabel("Total energy (J)")
ax2.set_title("Energy comparison (stacked)")
ax2.legend()
ax2.grid(axis="y", alpha=0.3)

plt.suptitle("Feeder Link vs ISL — cost comparison", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("scenario_comparison.png", dpi=120)
plt.show()
print("Saved scenario_comparison.png")

## 2. Space Exploration: ISL Speed Sweep

The ISL output module speed is swept across a range of values.
Three curves are compared on each plot:

| Curve | OBDH processing | ISL speed |
|---|---|---|
| **Base (fixed ISL)** | ✓ | fixed at scenario value |
| **With computing** | ✓ | variable |
| **Without computing** | ✗ | variable |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── Sweep parameters ──────────────────────────────────────────────────────────
isl_speeds = np.linspace(5e6, 100e6, 200)   # 1 Mbit/s → 1 Gbit/s

obdh = arch_B["OBDH"]
isl_fixed = arch_B["ISL"]   # fixed reference point

# Precompute quantities that don't change across the sweep
d_in   = d_in_compress
d_out  = d_out_compress
n_op   = algo_scenario.workloads["compress"]

map_t_fixed,  map_e_fixed  = mapping_cost({"compress": (obdh, d_in, n_op)})
rout_t_fixed, rout_e_fixed = routing_cost({("compress", "sink"): (isl_fixed, d_out)})
base_time   = np.full(200,time_A)
base_energy = np.full(200, energy_A)

# ── With computing: OBDH + variable ISL ───────────────────────────────────────
with_t, with_e = [], []
for s in isl_speeds:
    isl_var = OutputModule(energy=isl_fixed.energy, speed=s, link_length=isl_fixed.link_length)
    rt, re = routing_cost({("compress", "sink"): (isl_var, d_out)})
    with_t.append(map_t_fixed + rt)
    with_e.append(map_e_fixed + re)

# ── Without computing: no OBDH, raw data sent directly over variable ISL ──────
without_t, without_e = [], []
for s in isl_speeds:
    isl_var = OutputModule(energy=isl_fixed.energy, speed=s, link_length=isl_fixed.link_length)
    rt, re = routing_cost({("compress", "sink"): (isl_var, d_in)})   # d_in: uncompressed
    without_t.append(rt)
    without_e.append(re)

In [ ]:
isl_mbps = isl_speeds / 1e6   # convert to Mbit/s for readability

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# ── Energy ────────────────────────────────────────────────────────────────────
ax1.plot(isl_mbps, base_energy,   label="Base (fixed ISL)",    linewidth=2, color="gold",   linestyle="-")
ax1.plot(isl_mbps, with_e,        label="With computing",      linewidth=2, color="blue",   linestyle="-.")
ax1.plot(isl_mbps, without_e,     label="Without computing",   linewidth=2, color="red",    linestyle="--")
ax1.set_xlabel("ISL Speed (Mbit/s)")
ax1.set_ylabel("Energy Cost (J)")
ax1.set_title("Energy Budget", fontweight="bold")
ax1.set_xlim(isl_mbps[0], isl_mbps[-1])
ax1.grid(True, alpha=0.3)

# ── Time ──────────────────────────────────────────────────────────────────────
ax2.plot(isl_mbps, base_time,     label="Base (fixed ISL)",    linewidth=2, color="gold",   linestyle="-")
ax2.plot(isl_mbps, with_t,        label="With computing",      linewidth=2, color="blue",   linestyle="-.")
ax2.plot(isl_mbps, without_t,     label="Without computing",   linewidth=2, color="red",    linestyle="--")
ax2.set_xlabel("ISL Speed (Mbit/s)")
ax2.set_ylabel("Time Cost (s)")
ax2.set_title("Time Budget", fontweight="bold")
ax2.set_xlim(isl_mbps[0], isl_mbps[-1])
ax2.grid(True, alpha=0.3)

handles, labels = ax1.get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 1.05), ncol=3, fontsize=11)
plt.tight_layout()
plt.subplots_adjust(top=0.85)
plt.savefig("isl_sweep.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved isl_sweep.png")